In [ ]:
!nvidia-smi


In [ ]:
%cd /kaggle/working
!rm -rf yolov11s_kuangjing
!git clone https://github.com/tuanziya666/yolov11s_kuangjing.git
%cd /kaggle/working/yolov11s_kuangjing
!git checkout main
!git pull


In [ ]:
!python -c "from ultralytics import YOLO; YOLO('ultralytics/cfg/models/11/yolo11s-AKConv.yaml'); print('AKConv parse ok')"
!ls -lh train_yolo11s_akconv.py
!ls -lh ultralytics/cfg/models/11/yolo11s-AKConv.yaml


In [ ]:
from pathlib import Path

def find_dataset_roots(base: Path):
    roots = []
    for p in base.rglob('*'):
        if p.is_dir() and (p / 'images' / 'train').exists() and (p / 'labels' / 'train').exists():
            roots.append(p)
    return sorted(set(roots))

candidates = find_dataset_roots(Path('/kaggle/input'))
print('Candidates:')
for c in candidates:
    print(c)

DATASET_ROOT = candidates[0] if len(candidates) == 1 else Path('/kaggle/input/your-dataset-slug')
print('Using DATASET_ROOT =', DATASET_ROOT)


In [ ]:
from pathlib import Path

yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

yaml_path = Path('/kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml')
yaml_path.write_text(yaml_text, encoding='utf-8')
print(yaml_text)


In [ ]:
%cd /kaggle/working/yolov11s_kuangjing

RUN_NAME = 'yolo11s_akconv_kaggle'
EPOCHS = 100
BATCH = 16
WORKERS = 2
IMGSZ = 640
SEED = 42

!python -u train_yolo11s_akconv.py \
  --data ultralytics/cfg/datasets/coalmine4_kaggle.yaml \
  --name $RUN_NAME \
  --project /kaggle/working/runs \
  --device 0 \
  --epochs $EPOCHS \
  --batch $BATCH \
  --workers $WORKERS \
  --imgsz $IMGSZ \
  --seed $SEED


In [ ]:
%cd /kaggle/working/yolov11s_kuangjing

from ultralytics import YOLO

RUN_NAME = 'yolo11s_akconv_kaggle'
BEST = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'
print('BEST =', BEST)

model = YOLO(BEST)

model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_val',
    exist_ok=True,
)

model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)


In [ ]:
%cd /kaggle/working
!zip -qr yolo11s_akconv_kaggle_results.zip runs/yolo11s_akconv_kaggle evals/yolo11s_akconv_kaggle_val evals/yolo11s_akconv_kaggle_test
!ls -lh yolo11s_akconv_kaggle_results.zip
